# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. The dataset captures mental health indicators, demographic information, and assessment scores from survey responses collected in Kilifi County, Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object, not subscripting
metadata = dataset.metadata

print(f"Name: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here, we enumerate the available record sets in the dataset by their `@id`. For each record set, we display its fields (columns) and their corresponding `@id`s and names.

In [ ]:
# List all record sets by @id
record_set_infos = dataset.record_sets
record_set_ids = []

for rs in record_set_infos:
    print(f"Record Set @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    Field @id: {field['@id']} | Name: {field.get('name', 'N/A')}")
    elif 'columns' in rs:
        print("  Columns:")
        for col in rs['columns']:
            print(f"    Column @id: {col['@id']} | Name: {col.get('name', 'N/A')}")
    print("---")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

All entities (record sets, fields, columns, etc.) are referenced by their `@id` throughout.

In [ ]:
# Extract data from each record set
dataframes = {}

# If no record sets available, fallback
if len(record_set_ids) == 0:
    print("No record sets available in the schema.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records for Record Set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records and isinstance(records[0], dict):
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded dataframe with columns: {df.columns.tolist()}")
        else:
            print(f"No records found for {record_set_id} or inconsistent schema.")
    # Pick one of the available record sets for demonstration
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this section, we select a numeric field (referenced by its `@id`) for analysis. We filter records based on a threshold, normalize the field, and group the data by a key attribute (`@id`).

In [ ]:
# EDA on the primary record set
# Please replace the following @ids if your dataset has specific field ids, e.g. 'cr:GAD7_score' or similar

# Find a numeric field among columns or fields
df = dataframes[main_record_set_id]

# Attempt to auto-detect a numeric field (GAD-7, PHQ-9, PSQ) based on common names
numeric_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['gad', 'phq', 'psq', 'score'])]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]

print(f"Using numeric field: {numeric_field_id}")

# Set threshold for filtering
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to choose a grouping field (@id)
possible_group_fields = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'education', 'village', 'age', 'marital'])]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[0]
print(f"Grouping by field: {group_field}")
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we show a histogram for the key numeric field and a boxplot grouped by a demographic field, referenced by `@id`.

In [ ]:
# Visualization
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric score by group_field if group_field present and not numeric
if group_field in df.columns and df[group_field].dtype == object:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df[group_field], y=df[numeric_field_id])
plt.title(f"{numeric_field_id} by {group_field}")
plt.xticks(rotation=45)
plt.ylabel(numeric_field_id)
plt.xlabel(group_field)
plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² Mental Health Survey Dataset, explored its metadata and record sets with references by `@id`, extracted data for analysis, filtered and normalized numeric fields, and visualized key distributions. This approach provides a reproducible and FAIR-compliant workflow for preparing AI-ready datasets.

Further steps can include more detailed statistical analysis, modeling, or exporting processed data for downstream research tasks.